# 01 - Inventario y calidad de datos crudos

**Objetivo.** Cargar los datasets crudos desde `data/raw`, identificar estructura, tipos de datos, columnas, nulos, valores que funcionan como faltantes encubiertos, rangos de variables cuantitativas y cardinalidad de variables categóricas.

Este notebook no realiza limpieza fuerte. Su función es dejar una línea de base documentada para justificar las decisiones del notebook 02.

## Criterio de orden

- Los archivos crudos se leen desde rutas relativas al repositorio.
- Los crudos no se modifican.
- Airbnb se carga desde `airbnb_caba_normalizado_mensual.csv` como base prioritaria porque ya está armonizada para el análisis; los archivos de amenities/listings quedan como insumo de respaldo o enriquecimiento.
- Los outputs diagnósticos se guardan en `data/processed/reports/`.

In [ ]:
from pathlib import Path
import json
import re
import unicodedata
from datetime import datetime

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 80)
pd.set_option('display.width', 160)


def find_repo_root(start=None):
    """Busca la raíz del repo desde la ubicación actual del notebook.

    Funciona aunque el notebook se ejecute desde /notebooks, /data/processed
    o desde la raíz del proyecto.
    """
    start = Path(start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for path in candidates:
        if (path / 'data' / 'raw').exists() and (path / 'README.md').exists():
            return path
    raise FileNotFoundError(
        'No se encontró la raíz del repo. Ejecutar el notebook dentro del proyecto '
        'TP_analitica_descriptiva_grupo1_2q2026.'
    )

REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / 'data' / 'raw'
PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
REPORTS_DIR = PROCESSED_DIR / 'reports'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT:', REPO_ROOT)
print('RAW_DIR:', RAW_DIR)
print('PROCESSED_DIR:', PROCESSED_DIR)

## Carga de datasets crudos

La convención de nombres separa estado, fuente y operación:

- `df_raw_meli_alq`: Mercado Libre, alquiler permanente.
- `df_raw_meli_alq_temp`: Mercado Libre, alquiler temporario.
- `df_raw_meli_vtas`: Mercado Libre, ventas.
- `df_raw_argenprop`: Argenprop.
- `df_raw_zonaprop`: Zonaprop.
- `df_raw_airbnb_norm`: Airbnb normalizado mensual.

Esta convención se mantiene en el notebook 02 usando `df_preprocesado_*`.

In [ ]:
INVALID_MARKERS = {
    '', ' ', 'nan', 'NaN', 'NAN', 'none', 'None', 'NONE', 'null', 'NULL',
    's/d', 'S/D', 'sd', 'SD', 'sin dato', 'Sin dato', 'Sin Datos', 'sin datos',
    'no informa', 'No informa', 'no informado', 'No informado', 'n/a', 'N/A',
    '-', '--', '---', '?', '99999', '-99999', '-9999', '9999'
}

RAW_SOURCES = {
    'df_raw_meli_alq': {
        'descripcion': 'Mercado Libre - alquiler permanente',
        'paths': sorted((RAW_DIR / 'mercadolibre_inmuebles_alquiler').glob('meli_alq_*.csv')),
    },
    'df_raw_meli_alq_temp': {
        'descripcion': 'Mercado Libre - alquiler temporario',
        'paths': sorted((RAW_DIR / 'mercadolibre_inmuebles_alquiler temporario').glob('meli_alq_temp_*.csv')),
    },
    'df_raw_meli_vtas': {
        'descripcion': 'Mercado Libre - venta',
        'paths': sorted((RAW_DIR / 'mercadolibre_inmuebles_ventas').glob('meli_vtas_*.csv')),
    },
    'df_raw_argenprop': {
        'descripcion': 'Argenprop - publicaciones extraídas',
        'paths': [RAW_DIR / 'argenprop' / 'propiedades_argenprop.csv'],
    },
    'df_raw_zonaprop': {
        'descripcion': 'Zonaprop - publicaciones extraídas',
        'paths': [RAW_DIR / 'zonaprop' / 'propiedades_zonaprop.csv'],
    },
    'df_raw_airbnb_norm': {
        'descripcion': 'Airbnb CABA normalizado mensual. Base prioritaria para análisis comparable',
        'paths': [RAW_DIR / 'airbnb' / 'airbnb_caba_normalizado_mensual.csv'],
    },
}


def read_csv_safely(path):
    return pd.read_csv(path, low_memory=False)


def load_many(paths, source_name):
    existing = [Path(p) for p in paths if Path(p).exists()]
    if not existing:
        print(f'ATENCIÓN: no se encontraron archivos para {source_name}')
        return pd.DataFrame()
    frames = []
    for path in existing:
        df = read_csv_safely(path)
        df['_archivo_origen'] = str(path.relative_to(REPO_ROOT))
        frames.append(df)
    return pd.concat(frames, ignore_index=True, sort=False)

raw_dfs = {}
for df_name, spec in RAW_SOURCES.items():
    raw_dfs[df_name] = load_many(spec['paths'], df_name)
    globals()[df_name] = raw_dfs[df_name]
    print(f"{df_name}: {raw_dfs[df_name].shape} | {spec['descripcion']}")

## Inventario general

Control inicial de volumen, cantidad de columnas, memoria y fecha de extracción. Esto permite detectar cambios de cobertura entre fuentes y definir qué datasets tienen granularidad suficiente para la PreEntrega 2.

In [ ]:
COLUMN_DESCRIPTIONS = {
    'item_id': 'Identificador de publicación en Mercado Libre.',
    'property_id': 'Identificador de publicación o propiedad según fuente.',
    'fuente': 'Portal o fuente de origen del registro.',
    'url': 'URL de la publicación.',
    'search_url': 'URL de búsqueda desde la cual fue capturada la publicación.',
    'pagina_origen': 'Página de resultados donde apareció la publicación.',
    'scraped_at_utc': 'Fecha y hora de extracción en UTC.',
    'titulo': 'Título publicado del aviso.',
    'descripcion': 'Descripción textual del aviso cuando está disponible.',
    'tipo_propiedad': 'Tipo de inmueble declarado o inferido.',
    'tipo_operacion': 'Tipo de operación: venta, alquiler o alquiler temporal.',
    'moneda': 'Moneda del precio publicado.',
    'precio': 'Precio numérico extraído de la publicación.',
    'precio_texto': 'Precio en formato textual original.',
    'precio_m2_min': 'Precio por metro cuadrado mínimo calculado por Mercado Libre cuando hay rangos.',
    'precio_m2_max': 'Precio por metro cuadrado máximo calculado por Mercado Libre cuando hay rangos.',
    'precio_m2_calculado': 'Precio por metro cuadrado calculado por scraper en Argenprop/Zonaprop.',
    'barrio': 'Barrio declarado, inferido o capturado desde la fuente.',
    'barrio_norm': 'Barrio normalizado para Airbnb.',
    'comuna': 'Comuna de CABA cuando está disponible o inferida.',
    'localidad': 'Localidad declarada.',
    'provincia': 'Provincia o jurisdicción declarada.',
    'direccion': 'Dirección capturada en Mercado Libre.',
    'direccion_completa': 'Dirección capturada en Argenprop/Zonaprop/Airbnb.',
    'calle': 'Calle extraída o inferida.',
    'altura': 'Altura aproximada o declarada.',
    'latitud': 'Latitud de la publicación cuando la fuente la informa.',
    'longitud': 'Longitud de la publicación cuando la fuente la informa.',
    'ambientes_min': 'Mínimo de ambientes informado cuando la publicación usa rangos.',
    'ambientes_max': 'Máximo de ambientes informado cuando la publicación usa rangos.',
    'ambientes': 'Cantidad de ambientes.',
    'dormitorios_min': 'Mínimo de dormitorios informado cuando la publicación usa rangos.',
    'dormitorios_max': 'Máximo de dormitorios informado cuando la publicación usa rangos.',
    'dormitorios': 'Cantidad de dormitorios.',
    'banios_min': 'Mínimo de baños informado cuando la publicación usa rangos.',
    'banios_max': 'Máximo de baños informado cuando la publicación usa rangos.',
    'banios': 'Cantidad de baños.',
    'superficie_min_m2': 'Superficie mínima en m2 informada o inferida.',
    'superficie_max_m2': 'Superficie máxima en m2 informada o inferida.',
    'superficie_total_m2': 'Superficie total en m2.',
    'superficie_cubierta_m2': 'Superficie cubierta en m2.',
    'tipo_superficie': 'Tipo de superficie usada para el dato de m2.',
    'inmobiliaria': 'Nombre de inmobiliaria o anunciante cuando está disponible.',
    'seller_id': 'Identificador del vendedor/anfitrión cuando existe.',
    'seller_nombre': 'Nombre del anfitrión/vendedor en Airbnb.',
    'seller_superhost': 'Indicador de superhost en Airbnb.',
    'rating': 'Puntaje promedio de la publicación en Airbnb.',
    'reviews': 'Cantidad de reseñas en Airbnb.',
    'cantidad_resenas': 'Cantidad de reseñas en Airbnb no normalizado.',
    'cantidad_imagenes': 'Cantidad de imágenes detectadas.',
    'imagen_principal': 'URL de imagen principal.',
    'parse_warnings': 'Advertencias del proceso de parseo.',
    'parse_ok': 'Indicador de parseo exitoso.',
    'http_status': 'Código HTTP de respuesta durante scraping.',
    'apto_credito': 'Indicador de apto crédito.',
    'balcon': 'Indicador de balcón.',
    'balcon_o_patio': 'Indicador de balcón o patio en Airbnb.',
    'terraza': 'Indicador de terraza.',
    'pileta': 'Indicador de pileta.',
    'parrilla': 'Indicador de parrilla.',
    'cochera_mencionada': 'Indicador de cochera mencionada en Mercado Libre.',
    'cochera': 'Indicador de cochera en Airbnb/otros portales.',
    'amenities_mencionadas': 'Indicador de amenities mencionadas en Mercado Libre.',
    'precio_sospechoso': 'Indicador de precio sospechoso en Airbnb normalizado.',
    'unidad_precio': 'Unidad de referencia del precio Airbnb.',
    'precio_regimen': 'Régimen de precio Airbnb: por noche, mensual, etc.',
}


def example_value(series):
    sample = series.dropna().head(1)
    if sample.empty:
        return np.nan
    value = sample.iloc[0]
    text = str(value)
    return text[:120] + ('...' if len(text) > 120 else '')


def column_dictionary(df, dataset_name):
    rows = []
    for col in df.columns:
        rows.append({
            'dataset': dataset_name,
            'columna': col,
            'tipo_dato': str(df[col].dtype),
            'descripcion': COLUMN_DESCRIPTIONS.get(col, 'Pendiente de documentar según significado en la fuente.'),
            'ejemplo': example_value(df[col]),
        })
    return pd.DataFrame(rows)


def invalid_like_missing_count(series):
    if not (pd.api.types.is_object_dtype(series) or pd.api.types.is_string_dtype(series)):
        return 0
    normalized = series.astype('string').str.strip()
    return int(normalized.isin(INVALID_MARKERS).sum())


def usability_comment(null_pct, invalid_pct, n_labels, dtype):
    missing_pct = max(null_pct, invalid_pct)
    if missing_pct >= 80:
        return 'baja: faltantes muy altos; usar solo como referencia secundaria'
    if missing_pct >= 40:
        return 'media-baja: requiere justificación antes de usar en KPIs'
    if missing_pct >= 15:
        return 'media: usable con controles y posible segmentación'
    if n_labels > 300 and dtype == 'categorica':
        return 'media: alta cardinalidad; revisar si es identificador/texto libre'
    return 'alta: utilizable para análisis inicial'


def quality_profile(df, dataset_name):
    rows = []
    n = len(df)
    for col in df.columns:
        s = df[col]
        null_count = int(s.isna().sum())
        invalid_count = invalid_like_missing_count(s)
        is_num = pd.api.types.is_numeric_dtype(s)
        dtype_group = 'cuantitativa' if is_num else 'categorica/texto/fecha'
        numeric_min = s.min(skipna=True) if is_num and s.notna().any() else np.nan
        numeric_max = s.max(skipna=True) if is_num and s.notna().any() else np.nan
        n_labels = int(s.nunique(dropna=True)) if not is_num else np.nan
        null_pct = round(null_count / n * 100, 2) if n else np.nan
        invalid_pct = round(invalid_count / n * 100, 2) if n else np.nan
        rows.append({
            'dataset': dataset_name,
            'columna': col,
            'tipo_dato': str(s.dtype),
            'tipo_variable': dtype_group,
            'n_filas': n,
            'nulos_iniciales': null_count,
            'pct_nulos_iniciales': null_pct,
            'invalidos_tipo_faltante': invalid_count,
            'pct_invalidos_tipo_faltante': invalid_pct,
            'min_cuantitativa': numeric_min,
            'max_cuantitativa': numeric_max,
            'n_labels_categorica': n_labels,
            'comentario_usabilidad_calidad': usability_comment(null_pct, invalid_pct, 0 if pd.isna(n_labels) else n_labels, 'cuantitativa' if is_num else 'categorica'),
        })
    return pd.DataFrame(rows)


def dataset_inventory(dfs):
    rows = []
    for name, df in dfs.items():
        scraped_col = 'scraped_at_utc' if 'scraped_at_utc' in df.columns else None
        scraped_min = scraped_max = np.nan
        if scraped_col:
            dates = pd.to_datetime(df[scraped_col], errors='coerce', utc=True)
            if dates.notna().any():
                scraped_min = dates.min()
                scraped_max = dates.max()
        rows.append({
            'dataframe': name,
            'filas': len(df),
            'columnas': df.shape[1],
            'memoria_mb': round(df.memory_usage(deep=True).sum() / 1024**2, 2),
            'fecha_extraccion_min': scraped_min,
            'fecha_extraccion_max': scraped_max,
            'archivos_origen': df['_archivo_origen'].nunique() if '_archivo_origen' in df.columns else np.nan,
        })
    return pd.DataFrame(rows)

inventario_raw = dataset_inventory(raw_dfs)
display(inventario_raw)
inventario_raw.to_csv(REPORTS_DIR / 'inventario_raw.csv', index=False)

## Diccionario inicial de columnas

Tabla con nombre de columna, tipo de dato, descripción y ejemplo. Las descripciones desconocidas quedan marcadas como pendientes para completar manualmente luego de revisar el significado de la variable en la fuente.

In [ ]:
diccionarios = []
for name, df in raw_dfs.items():
    diccionarios.append(column_dictionary(df, name))

diccionario_raw_inicial = pd.concat(diccionarios, ignore_index=True)
display(diccionario_raw_inicial.head(80))
diccionario_raw_inicial.to_csv(PROCESSED_DIR / 'diccionario_raw_inicial.csv', index=False)
print('Guardado:', PROCESSED_DIR / 'diccionario_raw_inicial.csv')

## Revisión de columnas y tipos por dataset

Este bloque ayuda a verificar si las fuentes son directamente comparables o si requieren normalización de nombres, tipos y unidades.

In [ ]:
for name, df in raw_dfs.items():
    print('\n' + '=' * 100)
    print(name, df.shape)
    display(pd.DataFrame({
        'columna': df.columns,
        'tipo_dato': [str(df[c].dtype) for c in df.columns],
        'ejemplo': [example_value(df[c]) for c in df.columns],
    }))

## Perfil de calidad por columna

La tabla resume:

- conteo inicial de nulos;
- valores inválidos que probablemente representan faltantes (`s/d`, `sin datos`, `-99999`, etc.);
- rango de variables cuantitativas;
- cantidad de labels en variables categóricas;
- comentario preliminar de usabilidad/calidad.

La idea no es eliminar automáticamente registros, sino decidir qué necesita revisión en el notebook 02.

In [ ]:
perfiles = []
for name, df in raw_dfs.items():
    perfiles.append(quality_profile(df, name))

perfil_calidad_raw = pd.concat(perfiles, ignore_index=True)
perfil_calidad_raw = perfil_calidad_raw.sort_values(
    ['dataset', 'pct_nulos_iniciales', 'pct_invalidos_tipo_faltante'],
    ascending=[True, False, False]
)

display(perfil_calidad_raw.head(120))
perfil_calidad_raw.to_csv(PROCESSED_DIR / 'perfil_calidad_raw.csv', index=False)
print('Guardado:', PROCESSED_DIR / 'perfil_calidad_raw.csv')

## Variables con mayores problemas de completitud

Vista rápida de las columnas que más condicionan el uso posterior. Sirve para diferenciar variables centrales para KPIs de variables descriptivas secundarias.

In [ ]:
problemas_completitud = (
    perfil_calidad_raw
    .query('pct_nulos_iniciales >= 20 or pct_invalidos_tipo_faltante >= 5')
    .sort_values(['dataset', 'pct_nulos_iniciales', 'pct_invalidos_tipo_faltante'], ascending=[True, False, False])
)

display(problemas_completitud)
problemas_completitud.to_csv(REPORTS_DIR / 'problemas_completitud_raw.csv', index=False)

## Rangos de variables cuantitativas

Este control permite detectar outliers, errores de escala y campos que parecen numéricos pero representan flags o identificadores.

In [ ]:
rangos = []
for name, df in raw_dfs.items():
    num_cols = df.select_dtypes(include='number').columns
    if len(num_cols) == 0:
        continue
    tmp = df[num_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T.reset_index()
    tmp = tmp.rename(columns={'index': 'columna'})
    tmp.insert(0, 'dataset', name)
    rangos.append(tmp)

rangos_cuantitativos_raw = pd.concat(rangos, ignore_index=True) if rangos else pd.DataFrame()
display(rangos_cuantitativos_raw)
rangos_cuantitativos_raw.to_csv(REPORTS_DIR / 'rangos_cuantitativos_raw.csv', index=False)

## Cardinalidad y labels de variables categóricas

Se revisa cantidad de categorías y ejemplos. Las variables de alta cardinalidad pueden ser texto libre, URLs o identificadores; no conviene tratarlas como categorías analíticas sin procesarlas.

In [ ]:
cardinalidades = []
for name, df in raw_dfs.items():
    cat_cols = df.select_dtypes(include=['object', 'string', 'category', 'bool']).columns
    for col in cat_cols:
        vc = df[col].astype('string').str.strip().value_counts(dropna=False).head(10)
        cardinalidades.append({
            'dataset': name,
            'columna': col,
            'n_labels_incluyendo_na': int(df[col].nunique(dropna=False)),
            'top_10_labels': ' | '.join([f'{idx}: {val}' for idx, val in vc.items()])[:1000],
        })

cardinalidad_categoricas_raw = pd.DataFrame(cardinalidades).sort_values(
    ['dataset', 'n_labels_incluyendo_na'], ascending=[True, False]
)
display(cardinalidad_categoricas_raw.head(120))
cardinalidad_categoricas_raw.to_csv(REPORTS_DIR / 'cardinalidad_categoricas_raw.csv', index=False)

## Duplicados iniciales

Se evalúan duplicados exactos y duplicados por identificadores/URL cuando existen. No todos deben eliminarse automáticamente: puede haber republicaciones, unidades de emprendimientos o capturas en fechas distintas.

In [ ]:
keys_candidates = ['item_id', 'property_id', 'url']
rows = []
for name, df in raw_dfs.items():
    row = {
        'dataset': name,
        'filas': len(df),
        'duplicados_exactos': int(df.duplicated().sum()),
    }
    for key in keys_candidates:
        if key in df.columns:
            row[f'duplicados_por_{key}'] = int(df.duplicated(subset=[key]).sum())
            row[f'n_unicos_{key}'] = int(df[key].nunique(dropna=True))
    rows.append(row)

duplicados_raw = pd.DataFrame(rows)
display(duplicados_raw)
duplicados_raw.to_csv(REPORTS_DIR / 'duplicados_raw.csv', index=False)

## Comentarios de cierre del diagnóstico raw

Completar luego de ejecutar:

- Variables suficientemente confiables para usar en limpieza/consolidación:
- Variables centrales con faltantes importantes:
- Variables que requieren normalización de categorías:
- Posibles sesgos de captura por fuente:
- Decisiones que pasan al notebook 02:

> Nota: este diagnóstico no valida hipótesis. Solo define qué tan confiable es cada insumo para avanzar.

In [ ]:
# Escribir acá conclusiones breves luego de revisar las tablas anteriores.